## Gold Facts

In [0]:
%run ./00_config

In [0]:
from pyspark.sql import functions as F
from pyspark.sql import Window
from delta.tables import DeltaTable

In [0]:
latest_per_day = Window.partitionBy("video_id", "trending_dt").orderBy(
     F.col("_silver_processed_at").desc()
)

silver_df = (
    scoped_silver()
    .withColumn("rn", F.row_number().over(latest_per_day))
    .filter(F.col("rn") == 1)
    .drop("rn")
)

In [0]:
channels_df = spark.table(gold["dim_channel"]).select("channel_key", "channel_title")

video_history = Window.partitionBy("video_id").orderBy("trending_dt")

fact_src_df = (
    silver_df
    .join(channels_df, on="channel_title", how="left")
    .withColumn("views_delta", F.col("views") - F.lag("views").over(video_history))
    .withColumn("trending_day_num", F.row_number().over(video_history))
    .select(
        F.date_format("trending_dt", "yyyyMMdd").cast("int").alias("date_key"),
        "video_id",
        "channel_key",
        F.col("category_id").cast("int").alias("category_id"),
        "views",
        "likes",
        "dislikes",
        "comment_count",
        "views_delta",
        "trending_day_num",
    )
    .withColumn("_updated_at", F.current_timestamp())
)

In [0]:
checks = {
    "channel_key not found in dim_channel": fact_src_df.filter(F.col("channel_key").isNull()),
    "category_id not found in dim_category": fact_src_df.join(
        spark.table(gold["dim_category"]).select("category_id"), "category_id", "left_anti"),
    "date_key not found in dim_date": fact_src_df.join(
        spark.table(gold["dim_date"]).select("date_key"), "date_key", "left_anti"),
}

for problem, bad_rows in checks.items():
    if bad_rows.limit(1).count() > 0:
        display(bad_rows.limit(20))
        raise ValueError(f"Fact rows with {problem}")

In [0]:
target = DeltaTable.forName(spark, gold["fact_video_daily_stats"])

changed = """
    NOT (
        t.channel_key      <=> s.channel_key      AND
        t.category_id      <=> s.category_id      AND
        t.views            <=> s.views            AND
        t.likes            <=> s.likes            AND
        t.dislikes         <=> s.dislikes         AND
        t.comment_count    <=> s.comment_count    AND
        t.views_delta      <=> s.views_delta      AND
        t.trending_day_num <=> s.trending_day_num
    )
"""
(
    target.alias("t")
    .merge(
        fact_src_df.alias("s"),
        "t.video_id = s.video_id AND t.date_key = s.date_key"
    )
    .whenMatchedUpdateAll(condition=changed)
    .whenNotMatchedInsertAll()
    .whenNotMatchedBySourceDelete()
    .execute()
)

display(
    target.history(1)
    .select(
        "version",
        "timestamp",
        "operation",
        F.col("operationMetrics.numTargetRowsInserted").alias("inserted"),
        F.col("operationMetrics.numTargetRowsUpdated").alias("updated"),
    )
)